# LangChain Expression Language (LCEL)

In [ ]:
# 读取 .env 里的 OPENAI_API_KEY，供后面 LangChain 内部自动创建 OpenAI client 时使用
import os
import openai
from dotenv import load_dotenv,find_dotenv
_=load_dotenv(find_dotenv())
openai.api_key=os.environ["OPENAI_API_KEY"]

In [ ]:
# 【版本兼容修复】课程录制时（langchain<0.1）ChatPromptTemplate/ChatOpenAI/StrOutputParser
# 都在 langchain.prompts / langchain.chat_models / langchain.schema.output_parser 下面。
# 现在 langchain 1.x 把这些拆分到了独立的包：
#   - ChatPromptTemplate、StrOutputParser 等核心抽象搬到了 langchain_core
#   - ChatOpenAI 等具体模型集成搬到了 langchain_openai
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# prompt 模板：{topic} 是占位符，调用时会被实际值替换
prompt=ChatPromptTemplate.from_template(
    "tell me a short joke about {topic}"
)
model=ChatOpenAI()  # 默认聊天模型，会自动读取 OPENAI_API_KEY 环境变量
output_parser = StrOutputParser()  # 把模型返回的 AIMessage 对象解析成纯字符串

In [ ]:
# LCEL 的核心语法：用 | 把多个 Runnable 串联成一条 chain
# 数据流向：用户输入 -> prompt（拼成完整的消息）-> model（调用大模型）-> output_parser（解析成字符串）
chain = prompt | model | output_parser

In [ ]:
# invoke() 是所有 Runnable 的统一调用入口，传入一个 dict 来填充 prompt 里的变量
chain.invoke({"topic": "bears"})

## More complex chain

In [ ]:
# 【版本兼容修复】OpenAIEmbeddings 现在也在 langchain_openai 包里
# 【版本兼容修复】原课程用的 DocArrayInMemorySearch 依赖 docarray 这个包，当前环境没装 docarray，
# 会在 from_texts() 构造时直接 ImportError；这里换成同样是"进程内内存向量库"、
# 且已装好底层依赖（chromadb）的 Chroma，教学效果等价，用法几乎一致
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
# 把两句话文本转成向量并存入内存向量库；embedding 参数指定用什么模型做向量化
# 【版本兼容修复】DocArrayInMemorySearch -> Chroma（原因见上一个 cell 的注释）
vectorstore = Chroma.from_texts(
    ["harrison worked at kensho", "bears like to eat honey"],
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()  # 把向量库包装成检索器（Retriever），可以按相似度查询相关文档

In [ ]:
# 【版本兼容修复】get_relevant_documents() 在新版 langchain_core（BaseRetriever）里已经被彻底移除
# （不是 deprecation warning，是 AttributeError），统一改用 Runnable 通用接口 .invoke()
retriever.invoke("where did harrison work?")

In [ ]:
retriever.invoke("what do bears like to eat")

In [ ]:
# 这是一个 RAG（检索增强生成）的 prompt 模板：把检索到的 context 和用户问题一起交给模型
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# 【版本兼容修复】langchain.schema.runnable 已不存在，RunnableMap 现在在 langchain_core.runnables 下
# （RunnableMap 是 RunnableParallel 的别名，用于并行执行多个子任务、把结果合并成一个 dict）
from langchain_core.runnables import RunnableMap

In [ ]:
# RunnableMap 并行执行两个 lambda：一个负责检索 context，一个负责透传 question，
# 结果被合并成 {"context": [...], "question": "..."}，再交给 prompt 填充模板
# 【版本兼容修复】get_relevant_documents -> invoke
chain = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | model | output_parser

In [ ]:
chain.invoke({"question": "where did harrison work?"})

In [ ]:
# 单独把 RunnableMap 这一步拎出来看看它自己的输出是什么样的
# 【版本兼容修复】get_relevant_documents -> invoke
inputs = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
})

In [ ]:
inputs.invoke({"question": "where did harrison work?"})

In [ ]:
# 定义一个 function calling 用的函数 schema（和 L1 里的写法一样），
# 待会儿要用 model.bind(functions=...) 把它"绑定"到模型上
functions = [
    {
      "name": "weather_search",
      "description": "Search for weather given an airport code",
      "parameters": {
        "type": "object",
        "properties": {
          "airport_code": {
            "type": "string",
            "description": "The airport code to get the weather for"
          },
        },
        "required": ["airport_code"]
      }
    }
  ]

In [ ]:
# .bind() 是 LCEL 里"给 Runnable 预先绑定固定参数"的方法：
# 这里把 functions 这个 kwarg 固定绑定到 model 上，之后每次调用 model 都会自动带上这些函数定义，
# 不需要每次 invoke 时都手动传 functions 参数
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}")
    ]
)
model = ChatOpenAI(temperature=0).bind(functions=functions)

In [ ]:
runnable = prompt | model

In [ ]:
runnable.invoke({"input": "what is the weather in sf"})

In [ ]:
# 再加一个体育搜索函数，凑成两个可选函数，测试模型在多个函数里选择的能力
functions = [
    {
      "name": "weather_search",
      "description": "Search for weather given an airport code",
      "parameters": {
        "type": "object",
        "properties": {
          "airport_code": {
            "type": "string",
            "description": "The airport code to get the weather for"
          },
        },
        "required": ["airport_code"]
      }
    },
        {
      "name": "sports_search",
      "description": "Search for news of recent sport events",
      "parameters": {
        "type": "object",
        "properties": {
          "team_name": {
            "type": "string",
            "description": "The sports team to search for"
          },
        },
        "required": ["team_name"]
      }
    }
  ]

In [ ]:
# 重新 bind，用新的 functions 列表覆盖之前绑定的那个（原来的 model 对象没变，这里生成的是新对象）
model = model.bind(functions=functions)

In [ ]:
runnable = prompt | model

In [ ]:
# 这句问题跟 sports_search 更相关，期望模型会选择调用 sports_search 而不是 weather_search
runnable.invoke({"input": "how did the patriots do yesterday?"})

In [ ]:
# 【版本兼容修复】langchain.llms 已不存在，传统的补全型 LLM（非 Chat 模型）现在也在 langchain_openai 下
from langchain_openai import OpenAI
import json

In [ ]:
# gpt-3.5-turbo-instruct 是"补全"（completion）模型而不是"对话"（chat）模型，
# 它直接返回字符串而不是消息对象；这里刻意用一个较弱/较老的模型来演示后面 fallback 的必要性
simple_model = OpenAI(
    temperature=0,
    max_tokens=1000,
    model="gpt-3.5-turbo-instruct"
)
# 把模型输出直接接 json.loads：期望模型输出的文本本身就是合法 JSON，但这个假设并不总成立
simple_chain = simple_model | json.loads

In [ ]:
# 这个挑战性任务容易让 gpt-3.5-turbo-instruct 生成"看起来像 JSON 但格式不完全合法"的文本
# （比如漏引号、多余逗号等），导致 json.loads 解析失败——这正是后面要用 fallback 解决的问题
challenge = "write three poems in a json blob, where each poem is a json blob of a title, author, and first line"

In [ ]:
# 先看看原始模型输出的文本长什么样（还没经过 json.loads）
simple_model.invoke(challenge)

In [ ]:
# 这里经常会报 JSONDecodeError，因为 gpt-3.5-turbo-instruct 生成的文本不一定是严格合法的 JSON
simple_chain.invoke(challenge)

In [ ]:
# 换成更强的 Chat 模型（默认 gpt-3.5-turbo），通常更擅长生成格式规范的 JSON，
# 作为 simple_chain 失败时的备用链（fallback）
model = ChatOpenAI(temperature=0)
chain = model | StrOutputParser() | json.loads

In [ ]:
chain.invoke(challenge)

In [ ]:
# with_fallbacks：先尝试 simple_chain，如果它抛异常（比如 JSON 解析失败），
# 就自动改用 chain（更强的模型）重新跑一遍，对调用方是透明的
final_chain = simple_chain.with_fallbacks([chain])

In [ ]:
final_chain.invoke(challenge)

In [ ]:
# 重新构造一个最简单的 chain，用来演示 LCEL 统一接口：invoke / batch / stream / ainvoke
prompt = ChatPromptTemplate.from_template(
    "Tell me a short joke about {topic}"
)
model = ChatOpenAI()
output_parser = StrOutputParser()

chain = prompt | model | output_parser

In [ ]:
# invoke：单个输入，同步等待完整结果
chain.invoke({"topic": "bears"})

In [ ]:
# batch：一次性处理多个输入（LangChain 内部会做并发优化，比逐个 invoke 更快）
chain.batch([{"topic": "bears"}, {"topic": "frogs"}])

In [ ]:
# stream：流式输出，模型每生成一小块文本（token）就立刻 yield 出来，不用等全部生成完
for t in chain.stream({"topic": "bears"}):
    print(t)

In [ ]:
# ainvoke：invoke 的异步版本，在异步环境（比如 Web 服务端）中避免阻塞事件循环
# Jupyter 内核支持在 cell 里直接用 top-level await
response = await chain.ainvoke({"topic": "bears"})
response